# Supplementary Figure S3: analog characterisation (related to Figure 5)

Three panels, rendered as a single 6.5 x 8 in figure for Cell submission.

- **Panel A** - MSAs for the six inactive-prototype families. Row labels follow
  the Omega-mode-number convention (e.g. Ω-AT-5, Ω-AP-4). Template fully coloured
  by chemistry; analog residues coloured only where they match the template.
  Template names and lead analogs bolded.
- **Panel B** - *Transposed* MIC heatmap: strains on the y-axis (readable
  horizontal labels), all 65 peptides on the x-axis in family-ordered blocks.
  HC50 and CC50 as a narrow strip directly below the MIC heatmap. Colour scale
  is recentred at 8 μM with a stretched low-MIC segment so sub-micromolar cells
  render as distinctly darker burgundy than 1 μM cells.
- **Panel C** - Pooled conditioning-strategy comparison across all 59 analogs.
  Dots coloured by prototype family; lead analogs with a black marker edge.


In [ ]:
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from pathlib import Path
from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, LogNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

PROJECT = Path("/home/pszymczak/code/omegamp-dashboard/data")
OUT = Path("/home/pszymczak/code/omegamp-dashboard/figures")
OUT.mkdir(parents=True, exist_ok=True)

matplotlib.font_manager._load_fontmanager(try_read_cache=False)
plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "font.size": 6, "axes.titlesize": 6, "axes.labelsize": 6,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.6, "axes.edgecolor": "#333333",
    "xtick.color": "#333333", "ytick.color": "#333333",
    "axes.labelcolor": "#222222",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "figure.dpi": 300, "savefig.dpi": 600,
    "xtick.major.width": 0.5, "ytick.major.width": 0.5,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.major.pad": 2, "ytick.major.pad": 2,
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

## Load data and derive family assignments


In [ ]:
ref = pd.read_csv(PROJECT / "omegamp_reference_table.csv")
mic = pd.read_csv(PROJECT / "mic.csv")
hc50 = pd.read_csv(PROJECT / "hc50.csv")
cc50 = pd.read_csv(PROJECT / "cc50.csv")

# Prototype-column bookkeeping: two prototypes use DBAASPS IDs in the reference
PROTO_MAP = {"DBAASPS_20015": "OP-145-TII4", "DBAASPS_20955": "As-CATH4-6L"}

def resolve_family(row):
    if row["category"] == "prototype":
        return row["short_name"]
    p = str(row.get("prototype", ""))
    return PROTO_MAP.get(p, p) if p != "nan" else ""

ref["family"] = ref.apply(resolve_family, axis=1)
antimicrobial = ref[(ref["category"] == "analog")
                    & (ref["objective"] == "antimicrobial")].copy()
protos = ref[ref["category"] == "prototype"].set_index("family")

print(f"Prototypes:   {len(protos)}")
print(f"Antimicrobial analogs: {len(antimicrobial)}")


## Colours, leads, family ordering

`FC` matches Figure 5's family palette. `COND_COLOUR` is a separate palette for
AP/AT/AU. `LEADS_SET` contains the four analogs bolded throughout the figure.


In [ ]:
INACTIVE_ORDER = ["As-CATH4-6L", "Mammutin-1", "BoCo1",
                  "GQ20", "DeNo1047", "OP-145-TII4"]

# Family palette (matched to main Figure 5)
FC = {"Mammutin-1": "#D55E00", "GQ20": "#E69F00", "BoCo1": "#009E73",
      "DeNo1047": "#56B4E9", "As-CATH4-6L": "#0072B2",
      "OP-145-TII4": "#CC79A7"}

# Conditioning palette (distinct from family colours)
COND_COLOUR = {"AP": "#6a3d9a", "AT": "#1f78b4", "AU": "#33a02c"}

# Leads highlighted in Figure 5 and Table 1
LEADS_SET = {
    "\u03a9-AT-BoCo1-5", "\u03a9-AT-BoCo1-9",
    "\u03a9-AU-GQ20-4", "\u03a9-AP-Mammutin-1-4",
}

# ClustalX-style AA palette (matched to Figure S4)
AA_COLORS = {
    "A": "#87A7E0", "I": "#87A7E0", "L": "#87A7E0", "M": "#87A7E0",
    "F": "#87A7E0", "W": "#87A7E0", "V": "#87A7E0", "C": "#E787E0",
    "K": "#E87A7A", "R": "#E87A7A",
    "D": "#C256B0", "E": "#C256B0",
    "N": "#85C285", "Q": "#85C285", "S": "#85C285", "T": "#85C285",
    "G": "#F0A555", "P": "#D4C540",
    "H": "#5EC6C6", "Y": "#5EC6C6",
}


## Per-family peptide table

Returns `[prototype_row, analog_1, ..., analog_N]` ordered by conditioning then
by trailing numeric index.


In [ ]:
def family_rows(family):
    """Return [prototype_row, analog_row_1, ..., analog_row_N] for a family,
    sorted by conditioning then numeric index."""
    rows = []
    if family in protos.index:
        p = protos.loc[family].copy()
        p["family"] = family
        rows.append(p)
    sub = antimicrobial[antimicrobial["family"] == family].copy()
    # extract trailing integer from short_name for sort
    sub["idx"] = sub["short_name"].str.extract(r"-(\d+)$").astype(int)
    sub = sub.sort_values(["conditioning", "idx"])
    for _, r in sub.iterrows():
        rows.append(r)
    return rows


## Panel A: alignment drawing

Row labels use the Ω-mode-number convention (`Ω-AT-5`). `label_space` is set
dynamically from the longest row label so long family names like `As-CATH4-6L`
(11 characters) do not crop.


In [ ]:
def draw_alignment(ax, family, fontsize=6):
    rows = family_rows(family)
    labels = [family]
    seqs = [rows[0]["sequence"]]
    bolds = [True]
    for r in rows[1:]:
        # Analog label: keep conditioning visible (AP/AT/AU) and trailing index
        # e.g. 'Ω-AT-BoCo1-5' -> 'AT-5'
        parts = r["short_name"].split("-")
        cond = r["conditioning"]
        idx = parts[-1]
        labels.append(f"\u03a9-{cond}-{idx}")
        seqs.append(r["sequence"])
        bolds.append(r["short_name"] in LEADS_SET)

    template = seqs[0]
    max_len = max(len(s) for s in seqs)
    n = len(seqs)
    # Reserve enough left margin for the longest row label (usually the family
    # name, which can be up to 11 chars e.g. 'As-CATH4-6L').
    label_space = max(len(labels[0]), max(len(l) for l in labels[1:])) + 1.5

    for i, (lab, seq, b) in enumerate(zip(labels, seqs, bolds)):
        y = n - 1 - i
        is_template = (i == 0)
        weight = "bold" if b else "normal"
        ax.text(-0.3, y + 0.5, lab, ha="right", va="center",
                fontsize=fontsize, family="monospace", fontweight=weight)
        for j, aa in enumerate(seq):
            if is_template:
                color = AA_COLORS.get(aa, "white")
            else:
                color = AA_COLORS.get(aa, "white") if (j < len(template) and aa == template[j]) else "white"
            ax.add_patch(Rectangle((j, y), 1, 1, facecolor=color,
                                   edgecolor="none", linewidth=0))
            ax.text(j + 0.5, y + 0.5, aa, ha="center", va="center",
                    fontsize=fontsize, family="monospace", color="black",
                    fontweight=weight)

    # Family title (coloured)
    ax.text(max_len / 2, n + 0.6, family, ha="center", va="bottom",
            fontsize=fontsize + 1.5, fontweight="bold", color=FC[family])
    ax.set_xlim(-label_space, max_len + 0.3)
    ax.set_ylim(-0.3, n + 1.9)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


## Panel B: transposed MIC heatmap + HC50/CC50 strip

Strains on the y-axis (readable horizontal labels). All 65 peptides on the x-axis.
Cells with MIC > 64 μM are masked light gray (tested-but-inactive - rendered as
`NaN` in `mic.csv`). The MIC colour scale has a stretched low-MIC segment so
sub-μM cells render as a distinct dark burgundy, and `TwoSlopeNorm(vcenter=8)`
puts the red-to-blue transition at the standard AMP "active" threshold.

Deliberate departure from Figure 5E: this heatmap is transposed, so strain labels
are horizontal and the peptide axis is packed densely across the x. Figure 5E
labels every cell with a numeric MIC; here we rely on colour alone.


In [ ]:
# =========================================================================
STRAIN_LABELS = {
    "A. baumannii ATCC 19606 (-)": "$\\it{A. baumannii}$ 19606",
    "A. baumannii ATCC BAA-1605 (-) - CGTPACCIMRA":
        "$\\bf{\\it{A. baumannii}}$ $\\bf{BAA\\text{-}1605\\,(CRAB)}$",
    "E. cloacae ATCC 13047 (-)": "$\\it{E. cloacae}$ 13047",
    "E. coli ATCC 11775 (-)": "$\\it{E. coli}$ 11775",
    "E. coli AIC221 (-)": "$\\it{E. coli}$ AIC221",
    "E. coli AIC222 - CRE (-)":
        "$\\bf{\\it{E. coli}}$ $\\bf{AIC222\\,(CRE)}$",
    "E. coli ATCC BAA-3170 (-) - CRE":
        "$\\bf{\\it{E. coli}}$ $\\bf{BAA\\text{-}3170\\,(CRE)}$",
    "K. pneumoniae ATCC 13883 (-)": "$\\it{K. pneumoniae}$ 13883",
    "K. pneumoniae ATCC BAA-2342 (-) - EIRK":
        "$\\bf{\\it{K. pneumoniae}}$ $\\bf{BAA\\text{-}2342\\,(ESBL)}$",
    "P. aeruginosa PAO1 (-)": "$\\it{P. aeruginosa}$ PAO1",
    "P. aeruginosa PA14 (-)": "$\\it{P. aeruginosa}$ PA14",
    "P. aeruginosa ATCC BAA-3197 (-) - FBCRP":
        "$\\bf{\\it{P. aeruginosa}}$ $\\bf{BAA\\text{-}3197\\,(FQR)}$",
    "S. enterica ATCC 9150 (-)": "$\\it{S. enterica}$ 9150",
    "S. enterica Typhimurtium ATCC 700720": "$\\it{S. enterica}$ Typhi.",
    "B. subtilis ATCC 23857 (+)": "$\\it{B. subtilis}$ 23857",
    "S. aureus ATCC 12600 (+)": "$\\it{S. aureus}$ 12600",
    "S. aureus ATCC BAA-1556 - MRSA (+)":
        "$\\bf{\\it{S. aureus}}$ $\\bf{BAA\\text{-}1556\\,(MRSA)}$",
    "E. faecalis ATCC 700802 - VRE (+)":
        "$\\bf{\\it{E. faecalis}}$ $\\bf{700802\\,(VRE)}$",
    "E. faecium ATCC 700221 - VRE (+)":
        "$\\bf{\\it{E. faecium}}$ $\\bf{700221\\,(VRE)}$",
    "E. coli K-12 BW25113 (-)": "$\\it{E. coli}$ K-12",
}

mic_strains = list(mic.columns[1:])
# Same strain ordering as Figure 5: cluster the E. coli strains together
ecoli_strains = [s for s in mic_strains if "E. coli" in s or "coli" in s.lower()]
strain_order = []
for s in mic_strains:
    if s in ecoli_strains:
        continue
    strain_order.append(s)
    if "cloacae" in s.lower():
        strain_order.extend(ecoli_strains)
if not any("coli" in s.lower() for s in strain_order):
    strain_order.extend(ecoli_strains)

STRAIN_MDR = {s: any(t in s for t in ["CRAB", "CRE", "ESBL", "EIRK",
                                      "FBCR", "FQR", "MRSA", "VRE", "CGTPA"])
              for s in mic_strains}
STRAIN_GRAM = {s: ("+" if "(+)" in s else "-") for s in mic_strains}

# Colour maps
# MIC: diverging red-to-blue matched to Figure 5E in spirit, but with a
# stretched low-MIC segment so sub-micromolar cells render a distinct dark
# burgundy, and TwoSlopeNorm recentred at 8 uM (standard "active" threshold
# for AMP panels) so 0.125-1 uM occupies a meaningfully wide part of the scale.
MIC_CMAP = LinearSegmentedColormap.from_list(
    "mic_rb", [(0.0,  "#67000d"),  # sub-1 uM
              (0.08, "#b2182b"),   # ~1 uM
              (0.25, "#ef8a62"),
              (0.5,  "#f7f7f7"),   # at vcenter
              (0.75, "#67a9cf"),
              (1.0,  "#2166ac")])
MIC_NORM = TwoSlopeNorm(vmin=0, vcenter=8, vmax=64)

TOX_CMAP = LinearSegmentedColormap.from_list(
    "tox_gr", [(0.0, "#b2182b"), (0.2, "#ef8a62"), (0.4, "#fddbc7"),
              (0.5, "#f7f7f7"), (0.6, "#a6d96a"), (0.8, "#1a9641"),
              (1.0, "#006837")])
TOX_NORM = LogNorm(vmin=0.5, vmax=128)


In [ ]:
_GAP = "\x00"  # sentinel for family-separator columns

def draw_full_heatmap(ax_mic, ax_tox, cax_mic, cax_tox):
    """Transposed heatmap: strains on y-axis, peptides on x-axis.

    All cells — including untested / inactive (MIC ≥ 64) — are rendered
    purely by the MIC colormap after clipping to [0, 64]. No special
    grey overlays are applied; the top of the colour scale (dark blue)
    represents both 'tested-but-inactive' and 'not tested' cells.
    """
    mic_dict = {r["short_name"]: r for _, r in mic.iterrows()}
    hc50_dict = dict(zip(hc50["short_name"],
                         pd.to_numeric(hc50["HC50"], errors="coerce").clip(0.1, 128)))
    cc50_dict = dict(zip(cc50["short_name"],
                         pd.to_numeric(cc50["CC50"], errors="coerce").clip(0.1, 128)))

    # Peptides along x-axis, with one white-gap column inserted between families
    pep_names, pep_labels, pep_colors, pep_bold = [], [], [], []
    for fi, fam in enumerate(INACTIVE_ORDER):
        if fi > 0:                       # family separator
            pep_names.append(_GAP)
            pep_labels.append("")
            pep_colors.append("none")
            pep_bold.append(False)
        rows = family_rows(fam)
        for r in rows:
            nm = r["short_name"]
            pep_names.append(nm)
            if r.get("category") == "prototype":
                pep_labels.append(fam)
            else:
                idx = nm.split("-")[-1]
                pep_labels.append(f"\u03a9-{r['conditioning']}-{idx}")
            pep_colors.append(FC[fam])
            pep_bold.append(r.get("category") == "prototype" or nm in LEADS_SET)

    n_pep = len(pep_names)
    n_strains = len(strain_order)
    gap_cols = [j for j, nm in enumerate(pep_names) if nm == _GAP]

    # MIC matrix: default 128 (= missing = >64). Clip to [0,64] before imshow
    # so the colormap handles every cell uniformly.
    hm = np.full((n_strains, n_pep), 128.0)
    for j, nm in enumerate(pep_names):
        if nm == _GAP:
            continue
        if nm in mic_dict:
            for i, s in enumerate(strain_order):
                v = mic_dict[nm].get(s, np.nan)
                if pd.notna(v):
                    hm[i, j] = float(v)

    im_mic = ax_mic.imshow(hm.clip(0, 64), aspect="auto", cmap=MIC_CMAP,
                           norm=MIC_NORM, interpolation="nearest")

    # White separator columns (one tall rectangle per gap)
    for j in gap_cols:
        ax_mic.add_patch(Rectangle((j - 0.5, -0.5), 1, n_strains,
                                   facecolor="white", edgecolor="none", zorder=3))

    # Strain labels on the y-axis
    ax_mic.set_yticks(range(n_strains))
    ax_mic.tick_params(axis="y", which="major", pad=2, length=0)
    ylabs = ax_mic.set_yticklabels([STRAIN_LABELS.get(s, s) for s in strain_order],
                                   fontsize=6)
    for yi, s in enumerate(strain_order):
        ylabs[yi].set_color("#DC2626" if STRAIN_GRAM[s] == "+" else "#2563EB")
        if STRAIN_MDR[s]:
            ylabs[yi].set_fontweight("bold")
    ax_mic.set_xticks([])
    ax_mic.set_xlim(-0.5, n_pep - 0.5)

    # --- HC50 / CC50 strip ---
    tox_mat = np.full((2, n_pep), np.nan)
    for j, nm in enumerate(pep_names):
        if nm == _GAP:
            continue
        tox_mat[0, j] = hc50_dict.get(nm, np.nan)
        tox_mat[1, j] = cc50_dict.get(nm, np.nan)

    tox_ma = np.ma.array(tox_mat, mask=np.isnan(tox_mat))
    tox_cmap_masked = TOX_CMAP.copy()
    tox_cmap_masked.set_bad("#dddddd")
    im_tox = ax_tox.imshow(tox_ma, aspect="auto", cmap=tox_cmap_masked,
                           norm=TOX_NORM, interpolation="nearest")

    # White separators in tox strip
    for j in gap_cols:
        ax_tox.add_patch(Rectangle((j - 0.5, -0.5), 1, 2,
                                   facecolor="white", edgecolor="none", zorder=3))

    ax_tox.set_yticks([0, 1])
    ax_tox.set_yticklabels(["HC$_{50}$", "CC$_{50}$"], fontsize=6,
                           fontweight="bold")
    ax_tox.tick_params(axis="y", which="major", pad=2, length=0)

    # Peptide labels along x-axis of tox strip (90° rotation)
    ax_tox.set_xticks(range(n_pep))
    ax_tox.tick_params(axis="x", which="major", pad=2, length=0)
    xlabs = ax_tox.set_xticklabels(pep_labels, rotation=90, ha="right",
                                   fontsize=6, family="monospace")
    for xi, (c, b) in enumerate(zip(pep_colors, pep_bold)):
        xlabs[xi].set_color(c)
        if b:
            xlabs[xi].set_fontweight("bold")
    ax_tox.set_xlim(-0.5, n_pep - 0.5)

    # --- Colourbars ---
    cb_mic = plt.colorbar(im_mic, cax=cax_mic)
    cb_mic.set_label("MIC (μM)", fontsize=6)
    cb_mic.set_ticks([0, 1, 8, 16, 32, 64])
    cb_mic.ax.minorticks_off()
    cb_mic.ax.tick_params(labelsize=6)
    cb_mic.outline.set_visible(False)

    cb_tox = plt.colorbar(ScalarMappable(norm=TOX_NORM, cmap=TOX_CMAP),
                          cax=cax_tox)
    cb_tox.set_label("HC$_{50}$/CC$_{50}$ (μM)", fontsize=6, labelpad=2)
    cb_tox.set_ticks([1, 10, 100])
    cb_tox.set_ticklabels(["1", "10", "≥128"])
    cb_tox.ax.minorticks_off()
    cb_tox.ax.tick_params(labelsize=6)
    cb_tox.outline.set_visible(False)

## Panel C: pooled conditioning comparison

All 59 analogs on a single panel split by conditioning strategy (Ω-AP, Ω-AT,
Ω-AU). y-axis: number of strains with MIC ≤ 2 μM. Dots are coloured by prototype
family so the family-conditioning coupling (Mammutin-1 and OP-145-TII4 are
AP-only; As-CATH4-6L and BoCo1 are mostly AT; GQ20 is split AT/AU) is visible
rather than hidden.


In [ ]:
def draw_cond_panel(ax):
    mic_vals = mic[mic_strains].astype(float)
    n_le2_dict = dict(zip(mic["short_name"], (mic_vals <= 2).sum(axis=1)))

    sub = antimicrobial.copy()
    sub["n_le2"] = sub["short_name"].map(n_le2_dict).fillna(0)

    cond_order = ["AP", "AT", "AU"]
    for xi, cond in enumerate(cond_order):
        grp = sub[sub["conditioning"] == cond]
        if len(grp) == 0:
            continue
        med = grp["n_le2"].median()
        ax.plot([xi - 0.30, xi + 0.30], [med, med],
                color=COND_COLOUR[cond], lw=1.5, zorder=4)
        jit = np.random.RandomState(hash(cond) % 1000).normal(0, 0.10, len(grp))
        for j, (_, r) in enumerate(grp.iterrows()):
            ax.scatter(xi + jit[j], r["n_le2"],
                       c=FC[r["family"]],
                       s=18,
                       edgecolors="white",
                       linewidths=0.3,
                       alpha=0.9, zorder=3)
        ax.text(xi, 16.5, f"n={len(grp)}", ha="center", va="top",
                fontsize=6, color=COND_COLOUR[cond], fontweight="bold")

    ax.set_xticks(range(len(cond_order)))
    ax.set_xticklabels([f"$\\Omega$-{c}" for c in cond_order],
                       fontsize=6, fontweight="bold")
    for xl, c in zip(ax.get_xticklabels(), cond_order):
        xl.set_color(COND_COLOUR[c])
    ax.set_xlim(-0.5, len(cond_order) - 0.5)
    ax.set_ylim(-1, 17)
    ax.set_ylabel("Strains with MIC $\\leq$ 2 μM", fontsize=6)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

    # Family legend centred below the panel (avoids overlap with data)
    family_handles = [
        Line2D([], [], marker="o", color="w", markerfacecolor=FC[f],
               markeredgecolor="white", markeredgewidth=0.3, ms=5, lw=0,
               label=f)
        for f in INACTIVE_ORDER
    ]
    ax.legend(handles=family_handles, loc="upper center",
              bbox_to_anchor=(0.5, -0.10), ncol=3, fontsize=6, frameon=False,
              handletextpad=0.3, labelspacing=0.25, columnspacing=0.8,
              borderpad=0, title="Prototype family", title_fontsize=6)

## Assemble the full figure

Layout for the 6.5 x 8 inch figure:

- Panel A on top - 3 MSAs per row x 2 rows.
- Horizontal AA chemistry legend between Panel A and Panel B.
- Panel B in the middle - transposed heatmap, stacked MIC + HC50/CC50 strip,
  two colourbars on the right.
- Panel C at the bottom - pooled AP/AT/AU panel with family legend.

Minimum font size throughout: 6 pt.


In [ ]:
fig = plt.figure(figsize=(6.5, 6.8))

# Panel letter style
L = dict(fontsize=8, fontweight="bold", va="top", ha="left")

# ── Layout (figure fractions, bottom-origin, fig height = 6.8") ────────────
# All absolute heights identical to previous version except Panel C (0.88" = 50% shorter)
# and figure height reduced so bottom margin stays ~0.63".
#
#   top margin    0.10"
#   Panel A       2.00"   A_TOP=0.985  A_BOT=0.691
#   A→legend→B   0.40"
#   B MIC         1.76"   B_MIC_TOP=0.632  B_MIC_BOT=0.374
#   MIC→tox       0.03"
#   tox strip     0.25"   B_TOX_TOP=0.370  B_TOX_BOT=0.333
#   90°-labels    0.75"
#   Panel C       0.88"   C_TOP=0.222  C_BOT=0.093
#   bottom        0.63"
#   ─────────────────────────────────────────────────
#   Total         6.80" ✓

A_TOP, A_BOT    = 0.985, 0.691
B_MIC_TOP, B_MIC_BOT = 0.632, 0.374
B_TOX_TOP, B_TOX_BOT = 0.370, 0.333
C_TOP, C_BOT    = 0.222, 0.093

# -- Panel A ----------------------------------------------------------------
a_left, a_right = 0.02, 0.985
col_w = (a_right - a_left - 0.01 * 2) / 3
row_h = (A_TOP - A_BOT - 0.015) / 2
for idx, fam in enumerate(INACTIVE_ORDER):
    col = idx % 3
    row = idx // 3
    x0 = a_left + col * (col_w + 0.01)
    y0 = A_TOP - (row + 1) * row_h - row * 0.015
    ax = fig.add_axes([x0, y0, col_w, row_h])
    draw_alignment(ax, fam, fontsize=6)

fig.text(0.005, A_TOP, "A", **L)

# AA chemistry legend — 0.20" below Panel A bottom
legend_items = [
    ("Cationic (K,R)", "#E87A7A"),
    ("Anionic (D,E)", "#C256B0"),
    ("Hydrophobic", "#87A7E0"),
    ("Polar (S,T,N,Q)", "#85C285"),
    ("Glycine", "#F0A555"),
    ("Proline", "#D4C540"),
    ("His/Tyr", "#5EC6C6"),
]
handles = [Patch(facecolor=c, edgecolor="none", label=l) for l, c in legend_items]
fig.legend(handles=handles, loc="upper center",
           bbox_to_anchor=(0.5, 0.662),
           ncol=len(legend_items), fontsize=6, frameon=False,
           handletextpad=0.3, columnspacing=0.8,
           handlelength=0.9, handleheight=0.9, borderpad=0)

# -- Panel B ----------------------------------------------------------------
b_left, b_right = 0.22, 0.84
ax_mic = fig.add_axes([b_left, B_MIC_BOT, b_right - b_left, B_MIC_TOP - B_MIC_BOT])
ax_tox = fig.add_axes([b_left, B_TOX_BOT, b_right - b_left, B_TOX_TOP - B_TOX_BOT])
# Colourbars span the full height of the MIC heatmap
cax_mic = fig.add_axes([b_right + 0.022, B_MIC_BOT, 0.013, B_MIC_TOP - B_MIC_BOT])
cax_tox = fig.add_axes([b_right + 0.090, B_MIC_BOT, 0.013, B_MIC_TOP - B_MIC_BOT])
draw_full_heatmap(ax_mic, ax_tox, cax_mic, cax_tox)
fig.text(0.005, B_MIC_TOP + 0.005, "B", **L)

# -- Panel C (centered) --------------------------------------------------------
c_left, c_right = 0.25, 0.75
ax_c = fig.add_axes([c_left, C_BOT, c_right - c_left, C_TOP - C_BOT])
draw_cond_panel(ax_c)
fig.text(0.005, C_TOP + 0.005, "C", **L)
plt.show()

## Save


In [ ]:
out_png = OUT / "figureS3_analog-wet.png"
out_pdf = OUT / "figureS3_analog-wet.pdf"
out_svg = OUT / "figureS3_analog-wet.svg"
fig.savefig(out_png, dpi=400, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
fig.savefig(out_svg, bbox_inches="tight")
print(f"saved: {out_png}")
print(f"saved: {out_pdf}")
print(f"saved: {out_svg}")